# 🔬 Post-Training: ORPO
Aligns the SFT model using **ORPO**.

In [1]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "../01_sft/qwen_medical_arabic_lora",
    max_seq_length = 1024,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_training(model)
print("Model loaded for ORPO. VRAM:", round(torch.cuda.memory_allocated()/1e9, 2), "GB")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\Users\Lenovo\anaconda3\envs\unsloth_env\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


🦥 Unsloth Zoo will now patch everything to make training faster!


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit/resolve/main/config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 88e7e06a-b219-4bf2-95dc-c50ec2ac8758)')' thrown while requesting HEAD https://huggingface.co/unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit/resolve/main/config.json
[huggingface_hub.utils._http|WARNING]'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit/resolve/main/config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 88e7e06a-b219-4bf2-95dc-c50ec2ac8758)')' thrown while requesting HEAD https://huggingface.co/unslo

==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 3070 Ti Laptop GPU. Num GPUs = 1. Max memory: 8.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit/resolve/main/config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 2dafd27c-d354-4866-a347-ad3088ee69f3)')' thrown while requesting HEAD https://huggingface.co/unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit/resolve/main/config.json
[huggingface_hub.utils._http|WARNING]'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit/resolve/main/config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 2dafd27c-d354-4866-a347-ad3088ee69f3)')' thrown while requesting HEAD https://huggingface.co/unslo

Model loaded for ORPO. VRAM: 2.49 GB


In [2]:
from datasets import load_dataset
import os

DATA_FILE = "../../data/alignment/05_orpo_dataset.json"
assert os.path.exists(DATA_FILE), f"Data file not found: {DATA_FILE}"

dataset = load_dataset("json", data_files=DATA_FILE, split="train")
print(f"Dataset loaded: {len(dataset)} examples")

SYSTEM_MSG = "أنت معالج نفسي عربي خبير. ردودك آمنة ومتعاطفة وتراعي التعاليم الإسلامية السنية. لا تشخص ولا تصف أدوية."

def format_pair(example):
    prompt   = f"<|im_start|>system\n{SYSTEM_MSG}<|im_end|>\n<|im_start|>user\n{example['prompt']}<|im_end|>\n<|im_start|>assistant\n"
    chosen   = f"{example['chosen']}<|im_end|>\n"
    rejected = f"{example['rejected']}<|im_end|>\n"
    return {"prompt": prompt, "chosen": chosen, "rejected": rejected}

dataset = dataset.map(format_pair, remove_columns=dataset.column_names)
print(f"Formatted. Sample count: {len(dataset)}")
print("Sample prompt (first 150 chars):")
print(dataset[0]["prompt"][:150])


Dataset loaded: 300 examples
Formatted. Sample count: 300
Sample prompt (first 150 chars):
<|im_start|>system
أنت معالج نفسي عربي خبير. ردودك آمنة ومتعاطفة وتراعي التعاليم الإسلامية السنية. لا تشخص ولا تصف أدوية.<|im_end|>
<|im_start|>user
أ


In [3]:
from trl import ORPOTrainer, ORPOConfig

training_args = ORPOConfig(
    output_dir          = "outputs_orpo",
    beta                = 0.1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    learning_rate       = 5e-5,
    lr_scheduler_type   = "cosine",
    warmup_ratio        = 0.1,
    max_length          = 1024,
    max_prompt_length   = 512,
    optim               = "adamw_8bit",
    fp16                = not is_bfloat16_supported(),
    bf16                = is_bfloat16_supported(),
    num_train_epochs    = 2,
    logging_steps       = 10,
    save_steps          = 50,
    save_total_limit    = 2,
    report_to           = "none",
)

trainer = ORPOTrainer(
    model        = model,
    args         = training_args,
    train_dataset= dataset,
    tokenizer    = tokenizer,
)
print("ORPO Trainer ready!")


e:\FineTuning\model_training\02_post_training\unsloth_compiled_cache\UnslothORPOTrainer.py:763: UserWarning: This trainer will soon be moved to trl.experimental and is a candidate for removal. If you rely on it and want it to remain, please share your comments here: https://github.com/huggingface/trl/issues/4223. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  warnings.warn(
[trl.trainer.orpo_trainer|WARNING]When using DPODataCollatorWithPadding, you should set `remove_unused_columns=False` in your TrainingArguments we have set it for you, but you should do it yourself in the future.


ORPO Trainer ready!


In [4]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 300 | Num Epochs = 2 | Total steps = 74
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / rejected,logps / chosen,logits / rejected,logits / chosen,log_odds_ratio,log_odds_chosen,nll_loss
10,1.974700,-0.174912,-0.106732,0.012500,-0.068180,-1.067320,-1.749116,-1.983096,-2.422705,-1.273684,-0.923011,1.847358
20,1.483700,-0.149895,-0.115829,0.075000,-0.034066,-1.158293,-1.498954,-2.032475,-2.398627,-0.971955,-0.473059,1.386526
30,1.304800,-0.134663,-0.136043,0.537500,0.001381,-1.360433,-1.346627,-1.982528,-2.399674,-0.694304,0.024375,1.235398
40,1.164200,-0.128222,-0.145486,0.750000,0.017264,-1.454860,-1.282223,-1.960476,-2.350864,-0.596350,0.233795,1.165882
50,1.171500,-0.120859,-0.147646,0.900000,0.026788,-1.476462,-1.208587,-1.979148,-2.340782,-0.537412,0.364085,1.117739
60,1.148600,-0.119379,-0.153245,0.962500,0.033866,-1.532450,-1.193792,-1.979338,-2.367909,-0.499741,0.459080,1.098647
70,1.097500,-0.114410,-0.152148,0.975000,0.037737,-1.521475,-1.144102,-1.965463,-2.327792,-0.489151,0.514955,1.048586


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit/resolve/main/config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 8067bf0c-8a0c-43ff-bbe9-c573712fc72b)')' thrown while requesting HEAD https://huggingface.co/unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit/resolve/main/config.json
[huggingface_hub.utils._http|WARNING]'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit/resolve/main/config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 8067bf0c-8a0c-43ff-bbe9-c573712fc72b)')' thrown while requesting HEAD https://huggingface.co/unslo

TrainOutput(global_step=74, training_loss=1.325095679308917, metrics={'train_runtime': 2158.9597, 'train_samples_per_second': 0.278, 'train_steps_per_second': 0.034, 'total_flos': 0.0, 'train_loss': 1.325095679308917, 'epoch': 1.96})

In [ ]:
model.save_pretrained("qwen_medical_arabic_orpo")
tokenizer.save_pretrained("qwen_medical_arabic_orpo")
print("Saved ORPO model -> qwen_medical_arabic_orpo/")


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit/resolve/main/config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 5eb0a704-56f6-4ae9-8ba8-7a6899152721)')' thrown while requesting HEAD https://huggingface.co/unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit/resolve/main/config.json
[huggingface_hub.utils._http|WARNING]'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit/resolve/main/config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 5eb0a704-56f6-4ae9-8ba8-7a6899152721)')' thrown while requesting HEAD https://huggingface.co/unslo

Saved ORPO model -> qwen_medical_arabic_orpo/


: 